# 08 — Modeling Sequence: Baseline -> + Rate Level -> + Rate-of-Change/Interaction

**Purpose.** Runs the three-step comparison from `docs/phase0_framing.md`'s
Success Criteria table: a momentum-only baseline, then momentum + mortgage
rate level, then momentum + rate level + rate-of-change/interaction. The
baseline is CV-scored on the expanding-window folds from
`outputs/cv_folds.json` and locked in exactly once
(`outputs/baseline_lock.json`), so its own training/validation result
becomes the stated target rather than an arbitrary number fixed in advance.
Every marginal AUC delta gets a quarter-clustered bootstrap confidence
interval; a delta whose interval includes zero is reported as no
significant marginal contribution, not rounded to a small positive result.

**Model.** Logistic regression on standardized features -- a simple, fast
primary model that also lets the interaction term's fitted coefficient sign
be checked against `features.EXPECTED_INTERACTION_SIGN`
(`housing_risk.features`), which a tree-based model wouldn't support in the
same direct way.

**Business question (from `docs/phase0_framing.md`).**

> Can we predict, one quarter ahead, which US counties will see a housing
> price decline greater than 2%, and how much predictive power does
> mortgage-rate movement (30-year fixed rate level and rate-of-change) add
> once a county's own price momentum is already known?

This notebook only uses training-period cross-validation -- the held-out
test set (2022-01-01 to 2026-03-31) is never touched here. Final held-out
scoring is notebook 10's job, after both the primary and the robustness
(notebook 09) algorithm have been developed.


## 1. Setup

In [1]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from housing_risk import features

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Load the feature matrix and CV folds

Both are loaded from disk rather than recomputed, per this project's
convention that every later notebook builds on a previously persisted
result instead of silently drifting if it's regenerated: `outputs/cv_folds.json`
(notebook 05) and `../data/processed/train_features.csv` (notebook 07).


In [2]:
FEATURES_PATH = "../data/processed/train_features.csv"
FOLDS_PATH = "../outputs/cv_folds.json"

assert os.path.exists(FEATURES_PATH), (
    f"{FEATURES_PATH} not found -- run notebooks/07_feature_engineering.ipynb first, "
    "it persists the training-window feature matrix this notebook loads."
)
assert os.path.exists(FOLDS_PATH), (
    f"{FOLDS_PATH} not found -- run notebooks/05_split_cv.ipynb first, "
    "it persists the expanding-window CV fold boundaries this notebook loads."
)

matrix = pd.read_csv(FEATURES_PATH, parse_dates=["month"])
with open(FOLDS_PATH) as fh:
    folds = json.load(fh)
for fold in folds:
    for key in ("train_start", "train_end", "val_start", "val_end"):
        fold[key] = pd.Timestamp(fold[key])

print(f"Loaded {len(matrix):,} feature-matrix rows, {matrix['county_fips'].nunique():,} counties")
print(f"Loaded {len(folds)} CV folds ({folds[0]['train_start'].date()} .. {folds[-1]['val_end'].date()})")
matrix.head()


Loaded 164,460 feature-matrix rows, 3,009 counties
Loaded 11 CV folds (2000-03-31 .. 2020-12-31)


,county_fips,month,momentum_1m,momentum_3m,momentum_6m,mortgage_rate_level,mortgage_rate_roc,rate_roc_x_momentum,decline_label
0,1001,2001-03-31,0.004103,0.017537,0.031924,6.9520,-1.2880,-0.022588,0
1,1001,2001-06-30,-0.000412,0.002107,0.019681,7.1600,-1.1280,-0.002376,0
2,1001,2001-09-30,0.000778,-0.001648,0.000456,6.8175,-1.0945,0.001803,0
3,1001,2001-12-31,0.001478,0.008198,0.006537,7.0650,-0.3170,-0.002599,0
4,1001,2002-03-31,0.000517,-0.000605,0.007588,7.0140,0.0620,-0.000037,0


## 3. Feature steps and the fixed step order

Cumulative and ordered per `docs/phase0_framing.md`'s Field Scope: momentum
first, then what rate adds on top. The marginal lifts below are read
against this order -- they represent what rate contributes *given* momentum
is already known, not a claim about rate's importance in isolation.


In [3]:
MOMENTUM_FEATURES = ["momentum_1m", "momentum_3m", "momentum_6m"]
RATE_LEVEL_FEATURES = ["mortgage_rate_level"]
RATE_CHANGE_FEATURES = ["mortgage_rate_roc", "rate_roc_x_momentum"]

FEATURE_STEPS = {
    "baseline": MOMENTUM_FEATURES,
    "step2_rate_level": MOMENTUM_FEATURES + RATE_LEVEL_FEATURES,
    "step3_rate_change": MOMENTUM_FEATURES + RATE_LEVEL_FEATURES + RATE_CHANGE_FEATURES,
}
assert FEATURE_STEPS["step3_rate_change"] == features.FEATURE_COLUMNS, (
    "STOP: this notebook's step-3 feature list has drifted from housing_risk.features.FEATURE_COLUMNS"
)
for name, cols in FEATURE_STEPS.items():
    print(f"{name}: {cols}")

TARGET = "decline_label"


baseline: ['momentum_1m', 'momentum_3m', 'momentum_6m']
step2_rate_level: ['momentum_1m', 'momentum_3m', 'momentum_6m', 'mortgage_rate_level']
step3_rate_change: ['momentum_1m', 'momentum_3m', 'momentum_6m', 'mortgage_rate_level', 'mortgage_rate_roc', 'rate_roc_x_momentum']


## 4. Minimum positives per CV fold (gate, per `docs/phase0_framing.md`)

Any CV fold used for scoring needs a minimum number of positive-class
examples in its validation window before its AUC is trusted -- a fold with
too few positives produces an unstable or undefined AUC and can distort
model selection. This must run automatically, since the early years of the
training window have naturally lower decline rates. `MIN_FOLD_POSITIVES` is
this notebook's own choice (not specified in `docs/phase0_framing.md`);
revisit if real fold sizes make it look too loose or too strict.


In [4]:
MIN_FOLD_POSITIVES = 20  # my choice, not specified in docs/phase0_framing.md

def fold_val_mask(fold):
    return (matrix["month"] >= fold["val_start"]) & (matrix["month"] <= fold["val_end"])

fold_status = []
for fold in folds:
    val_rows = matrix.loc[fold_val_mask(fold)]
    n_positive = int(val_rows[TARGET].sum())
    status = "ok" if n_positive >= MIN_FOLD_POSITIVES else "skip"
    fold_status.append({**fold, "n_val_rows": len(val_rows), "n_val_positives": n_positive, "status": status})

fold_status_df = pd.DataFrame(fold_status)
print(fold_status_df[["fold_id", "val_start", "val_end", "n_val_rows", "n_val_positives", "status"]])

ok_folds = [f for f in fold_status if f["status"] == "ok"]
skipped_folds = [f for f in fold_status if f["status"] == "skip"]
print(f"\n{len(ok_folds)} of {len(folds)} folds usable (>= {MIN_FOLD_POSITIVES} validation positives)")
if skipped_folds:
    print(f"Skipped folds (below the minimum, excluded from all scoring below): "
          f"{[f['fold_id'] for f in skipped_folds]}")
assert len(ok_folds) >= 3, (
    "STOP: fewer than 3 usable CV folds after the minimum-positives gate -- too few folds "
    "to trust a CV-based baseline lock. Investigate MIN_FOLD_POSITIVES or the label distribution."
)


    fold_id  val_start    val_end  n_val_rows  n_val_positives status
0         0 2010-03-31 2010-12-31        8023             1130     ok
1         1 2011-03-31 2011-12-31        8308              789     ok
2         2 2012-03-31 2012-12-31        8963              341     ok
3         3 2013-03-31 2013-12-31        9728              272     ok
4         4 2014-03-31 2014-12-31        9926              315     ok
5         5 2015-03-31 2015-12-31       10082              120     ok
6         6 2016-03-31 2016-12-31       11024              373     ok
7         7 2017-03-31 2017-12-31       11895              255     ok
8         8 2018-03-31 2018-12-31       11918              317     ok
9         9 2019-03-31 2019-12-31       11966              131     ok
10       10 2020-03-31 2020-12-31       11998               79     ok

11 of 11 folds usable (>= 20 validation positives)


## 5. Cross-validated scoring, per feature step

For each usable fold, a fresh model is trained on that fold's training
window and scored on its validation window -- standardization is fit on
the fold's training data only (inside the `Pipeline`), so no validation-fold
information leaks into the scaler. Both a per-fold mean/std (the
conventional CV summary) and pooled out-of-fold (OOF) predictions
(concatenated across all usable folds) are produced: the pooled OOF set is
what Section 7's quarter-clustered bootstrap resamples from, since a
bootstrap needs an actual population of rows to resample, not a per-fold
summary statistic.


In [5]:
def make_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
    ])

def cv_score_step(feature_cols, ok_folds=ok_folds):
    per_fold_rows = []
    oof_rows = []
    for fold in ok_folds:
        train_mask = matrix["month"] <= fold["train_end"]
        val_mask = fold_val_mask(fold)
        train_df = matrix.loc[train_mask]
        val_df = matrix.loc[val_mask]

        model = make_model()
        model.fit(train_df[feature_cols], train_df[TARGET])
        val_pred = model.predict_proba(val_df[feature_cols])[:, 1]

        fold_roc_auc = roc_auc_score(val_df[TARGET], val_pred)
        fold_pr_auc = average_precision_score(val_df[TARGET], val_pred)
        fold_prevalence = float(val_df[TARGET].mean())
        per_fold_rows.append({
            "fold_id": fold["fold_id"], "roc_auc": fold_roc_auc, "pr_auc": fold_pr_auc,
            "prevalence": fold_prevalence, "n_val": len(val_df),
        })
        oof_rows.append(pd.DataFrame({
            "fold_id": fold["fold_id"], "month": val_df["month"].to_numpy(),
            "y_true": val_df[TARGET].to_numpy(), "y_pred": val_pred,
        }))

    per_fold = pd.DataFrame(per_fold_rows)
    oof = pd.concat(oof_rows, ignore_index=True)
    pooled_roc_auc = roc_auc_score(oof["y_true"], oof["y_pred"])
    pooled_pr_auc = average_precision_score(oof["y_true"], oof["y_pred"])
    pooled_prevalence = float(oof["y_true"].mean())
    summary = {
        "mean_roc_auc": float(per_fold["roc_auc"].mean()),
        "std_roc_auc": float(per_fold["roc_auc"].std()),
        "mean_pr_auc": float(per_fold["pr_auc"].mean()),
        "std_pr_auc": float(per_fold["pr_auc"].std()),
        "pooled_roc_auc": float(pooled_roc_auc),
        "pooled_pr_auc": float(pooled_pr_auc),
        "pooled_prevalence": pooled_prevalence,
        "n_folds": len(ok_folds),
    }
    return summary, per_fold, oof

step_summaries = {}
step_per_fold = {}
step_oof = {}
for step_name, cols in FEATURE_STEPS.items():
    summary, per_fold, oof = cv_score_step(cols)
    step_summaries[step_name] = summary
    step_per_fold[step_name] = per_fold
    step_oof[step_name] = oof
    print(f"{step_name}: mean ROC-AUC={summary['mean_roc_auc']:.4f} (+/-{summary['std_roc_auc']:.4f}), "
          f"pooled ROC-AUC={summary['pooled_roc_auc']:.4f}, pooled PR-AUC={summary['pooled_pr_auc']:.4f} "
          f"(pooled prevalence={summary['pooled_prevalence']:.4f})")


baseline: mean ROC-AUC=0.8536 (+/-0.0276), pooled ROC-AUC=0.8873, pooled PR-AUC=0.4139 (pooled prevalence=0.0362)
step2_rate_level: mean ROC-AUC=0.8529 (+/-0.0286), pooled ROC-AUC=0.8858, pooled PR-AUC=0.4084 (pooled prevalence=0.0362)
step3_rate_change: mean ROC-AUC=0.8532 (+/-0.0295), pooled ROC-AUC=0.8854, pooled PR-AUC=0.4062 (pooled prevalence=0.0362)


## 6. Lock the baseline (once)

Per `docs/phase0_framing.md`'s Success Criteria table, the baseline has no
target fixed in advance -- it's trained first, scored on training/validation
data, and that result becomes the stated target for step 2. Locked to
`outputs/baseline_lock.json` **exactly once**: if the file already exists,
this cell loads and reports it instead of re-locking, so re-running this
notebook can't silently move the target after the fact.


In [6]:
BASELINE_LOCK_PATH = "../outputs/baseline_lock.json"

if os.path.exists(BASELINE_LOCK_PATH):
    with open(BASELINE_LOCK_PATH) as fh:
        baseline_lock = json.load(fh)
    print(f"Baseline already locked at {BASELINE_LOCK_PATH} -- skipping re-lock.")
    print(f"Locked baseline pooled ROC-AUC: {baseline_lock['pooled_roc_auc']:.4f} "
          f"(mean CV ROC-AUC: {baseline_lock['mean_roc_auc']:.4f} +/- {baseline_lock['std_roc_auc']:.4f})")
else:
    baseline_summary = step_summaries["baseline"]
    baseline_lock = {
        **baseline_summary,
        "feature_columns": FEATURE_STEPS["baseline"],
        "cv_folds_path": FOLDS_PATH,
        "min_fold_positives": MIN_FOLD_POSITIVES,
        "n_folds_available": len(folds),
        "n_folds_used": len(ok_folds),
        "model": "LogisticRegression(standardized)",
        "random_state": RANDOM_STATE,
    }
    os.makedirs(os.path.dirname(BASELINE_LOCK_PATH), exist_ok=True)
    with open(BASELINE_LOCK_PATH, "w") as fh:
        json.dump(baseline_lock, fh, indent=2)
    print(f"Baseline locked and written to {BASELINE_LOCK_PATH}.")
    print(f"Locked baseline pooled ROC-AUC: {baseline_lock['pooled_roc_auc']:.4f} "
          f"(mean CV ROC-AUC: {baseline_lock['mean_roc_auc']:.4f} +/- {baseline_lock['std_roc_auc']:.4f})")

assert baseline_lock["pooled_roc_auc"] > 0.5, (
    "Sanity note (not a success criterion): the locked baseline's pooled ROC-AUC is at or "
    "below 0.5 -- investigate before treating step 2/3 deltas against it as meaningful."
)


Baseline already locked at ../outputs/baseline_lock.json -- skipping re-lock.
Locked baseline pooled ROC-AUC: 0.8873 (mean CV ROC-AUC: 0.8536 +/- 0.0276)


## 7. Quarter-clustered bootstrap on marginal AUC deltas

Per `docs/phase0_framing.md`'s Model Validation Requirements, confidence
intervals on every AUC delta use a cluster bootstrap that resamples whole
quarters (not individual county-quarter rows and not individual counties):
counties within a single quarter share the same national rate environment
and overlapping regional conditions, so they are not independent
observations of each other. Each bootstrap draw resamples the set of
distinct validation quarters (in the pooled OOF set built in Section 5) with
replacement, keeping every row for a chosen quarter together, and reuses the
exact same quarter draws for both models being compared -- a paired
comparison on identical resampled rows, so the delta's uncertainty reflects
only the two models' disagreement, not sampling noise from using different
rows for each.


In [7]:
N_BOOTSTRAP = 2000  # my choice, not specified in docs/phase0_framing.md
rng = np.random.default_rng(RANDOM_STATE)

def quarter_clustered_bootstrap_delta(oof_a, oof_b, n_bootstrap=N_BOOTSTRAP, rng=rng):
    """Bootstrap CI on the ROC-AUC delta (oof_b - oof_a), resampling whole
    quarters with replacement. oof_a/oof_b must share the same `month`
    values row-for-row (same folds, same rows) so the resample is paired."""
    assert oof_a["month"].equals(oof_b["month"]), "oof_a/oof_b must be row-aligned on the same validation rows"
    assert (oof_a["y_true"].to_numpy() == oof_b["y_true"].to_numpy()).all(), "y_true must match row-for-row"

    df = pd.DataFrame({
        "month": oof_a["month"].to_numpy(),
        "y_true": oof_a["y_true"].to_numpy(),
        "pred_a": oof_a["y_pred"].to_numpy(),
        "pred_b": oof_b["y_pred"].to_numpy(),
    })
    quarters = df["month"].unique()
    by_quarter = {q: df.loc[df["month"] == q] for q in quarters}

    deltas = []
    for _ in range(n_bootstrap):
        draw = rng.choice(quarters, size=len(quarters), replace=True)
        resampled = pd.concat([by_quarter[q] for q in draw], ignore_index=True)
        if resampled["y_true"].nunique() < 2:
            continue  # a resample with only one class can't score ROC-AUC; skip, don't crash
        auc_a = roc_auc_score(resampled["y_true"], resampled["pred_a"])
        auc_b = roc_auc_score(resampled["y_true"], resampled["pred_b"])
        deltas.append(auc_b - auc_a)

    deltas = np.array(deltas)
    point_delta = roc_auc_score(df["y_true"], df["pred_b"]) - roc_auc_score(df["y_true"], df["pred_a"])
    ci_low, ci_high = float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))
    significant = not (ci_low <= 0.0 <= ci_high)
    return {
        "point_delta": float(point_delta),
        "ci_95_low": ci_low,
        "ci_95_high": ci_high,
        "n_bootstrap_used": int(len(deltas)),
        "significant": bool(significant),
    }

delta_2_vs_1 = quarter_clustered_bootstrap_delta(step_oof["baseline"], step_oof["step2_rate_level"])
delta_3_vs_2 = quarter_clustered_bootstrap_delta(step_oof["step2_rate_level"], step_oof["step3_rate_change"])

print("Step 2 (+ rate level) vs. baseline, ROC-AUC delta:")
print(f"  point estimate: {delta_2_vs_1['point_delta']:+.4f}  "
      f"95% CI: [{delta_2_vs_1['ci_95_low']:+.4f}, {delta_2_vs_1['ci_95_high']:+.4f}]")
print("Step 3 (+ rate-of-change/interaction) vs. step 2, ROC-AUC delta:")
print(f"  point estimate: {delta_3_vs_2['point_delta']:+.4f}  "
      f"95% CI: [{delta_3_vs_2['ci_95_low']:+.4f}, {delta_3_vs_2['ci_95_high']:+.4f}]")


Step 2 (+ rate level) vs. baseline, ROC-AUC delta:
  point estimate: -0.0016  95% CI: [-0.0039, +0.0012]
Step 3 (+ rate-of-change/interaction) vs. step 2, ROC-AUC delta:
  point estimate: -0.0004  95% CI: [-0.0017, +0.0009]


## 8. Success criteria check

Per `docs/phase0_framing.md`'s Success Criteria table:

- **Step 2** (+ rate level): ROC-AUC >= baseline + 0.01.
- **Step 3** (+ rate-of-change/interaction): ROC-AUC >= step 2 + 0.01, or a
  stated null result.

A delta whose 95% CI includes zero is reported as "no significant marginal
contribution", never rounded to a small positive result, regardless of
whether the point estimate alone happens to clear the +0.01 threshold.


In [8]:
def report_criterion(label, delta_result, threshold=0.01):
    meets_point_threshold = delta_result["point_delta"] >= threshold
    print(f"{label}:")
    print(f"  point delta {delta_result['point_delta']:+.4f} vs. threshold +{threshold:.2f}: "
          f"{'meets' if meets_point_threshold else 'does not meet'} the point-estimate bar")
    if not delta_result["significant"]:
        print(f"  95% CI [{delta_result['ci_95_low']:+.4f}, {delta_result['ci_95_high']:+.4f}] includes zero "
              f"-> NO SIGNIFICANT MARGINAL CONTRIBUTION (reported as a null result, not rounded to a small positive one)")
    else:
        direction = "positive" if delta_result["point_delta"] > 0 else "negative"
        print(f"  95% CI [{delta_result['ci_95_low']:+.4f}, {delta_result['ci_95_high']:+.4f}] excludes zero "
              f"-> statistically significant {direction} delta")

report_criterion("Step 2 (+ rate level) vs. baseline", delta_2_vs_1)
print()
report_criterion("Step 3 (+ rate-of-change/interaction) vs. step 2", delta_3_vs_2)


Step 2 (+ rate level) vs. baseline:
  point delta -0.0016 vs. threshold +0.01: does not meet the point-estimate bar
  95% CI [-0.0039, +0.0012] includes zero -> NO SIGNIFICANT MARGINAL CONTRIBUTION (reported as a null result, not rounded to a small positive one)

Step 3 (+ rate-of-change/interaction) vs. step 2:
  point delta -0.0004 vs. threshold +0.01: does not meet the point-estimate bar
  95% CI [-0.0017, +0.0009] includes zero -> NO SIGNIFICANT MARGINAL CONTRIBUTION (reported as a null result, not rounded to a small positive one)


## 9. Interaction coefficient sign check

`housing_risk.features.EXPECTED_INTERACTION_SIGN` states the theoretical
sign (`'negative'`) for `rate_roc_x_momentum` under this project's plain-
product coding. A single logistic regression is refit here on the *full*
usable training-period data (all rows covered by the usable CV folds'
training windows, i.e. through the last fold's training end) with the full
step-3 feature set, and its fitted `rate_roc_x_momentum` coefficient's sign
is checked against that stated expectation -- a mismatch is reported as a
face-validity flag, not smoothed over.


In [9]:
full_train_df = matrix.loc[matrix["month"] <= folds[-1]["train_end"]]
sign_check_model = make_model()
sign_check_model.fit(full_train_df[FEATURE_STEPS["step3_rate_change"]], full_train_df[TARGET])

coefs = pd.Series(
    sign_check_model.named_steps["clf"].coef_[0],
    index=FEATURE_STEPS["step3_rate_change"],
)
print("Fitted logistic-regression coefficients (standardized features):")
print(coefs)

interaction_coef = coefs["rate_roc_x_momentum"]
observed_sign = "negative" if interaction_coef < 0 else "positive"
expected_sign = features.EXPECTED_INTERACTION_SIGN
sign_matches = observed_sign == expected_sign

print(f"\nrate_roc_x_momentum coefficient: {interaction_coef:+.4f}")
print(f"Expected sign: {expected_sign}  |  Observed sign: {observed_sign}  |  Match: {sign_matches}")
if not sign_matches:
    print(
        "FACE-VALIDITY FLAG: the fitted interaction coefficient contradicts the stated theory "
        "(housing_risk.features docstring) -- report as a surprise to investigate, not smoothed over. "
        "Note notebook 06/07 already flagged that the winning rate-of-change lag's own correlation "
        "sign didn't match theory either, so this would not be a first surprise in this pipeline."
    )


Fitted logistic-regression coefficients (standardized features):
momentum_1m           -2.841749
momentum_3m            1.909807
momentum_6m           -1.113508
mortgage_rate_level    0.310286
mortgage_rate_roc     -0.092049
rate_roc_x_momentum    0.192025
dtype: float64

rate_roc_x_momentum coefficient: +0.1920
Expected sign: negative  |  Observed sign: positive  |  Match: False
FACE-VALIDITY FLAG: the fitted interaction coefficient contradicts the stated theory (housing_risk.features docstring) -- report as a surprise to investigate, not smoothed over. Note notebook 06/07 already flagged that the winning rate-of-change lag's own correlation sign didn't match theory either, so this would not be a first surprise in this pipeline.


## 10. Persist the modeling-sequence results

In [10]:
results = {
    "feature_steps": FEATURE_STEPS,
    "step_summaries": step_summaries,
    "min_fold_positives": MIN_FOLD_POSITIVES,
    "n_folds_used": len(ok_folds),
    "n_folds_skipped": len(skipped_folds),
    "skipped_fold_ids": [f["fold_id"] for f in skipped_folds],
    "delta_step2_vs_baseline": delta_2_vs_1,
    "delta_step3_vs_step2": delta_3_vs_2,
    "interaction_coefficient": float(interaction_coef),
    "interaction_expected_sign": expected_sign,
    "interaction_observed_sign": observed_sign,
    "interaction_sign_matches": bool(sign_matches),
    "n_bootstrap": N_BOOTSTRAP,
    "random_state": RANDOM_STATE,
}
os.makedirs("../outputs", exist_ok=True)
with open("../outputs/modeling_sequence_logreg.json", "w") as fh:
    json.dump(results, fh, indent=2)
print("Saved outputs/modeling_sequence_logreg.json")


Saved outputs/modeling_sequence_logreg.json


## 11. Findings

- **Baseline locked exactly once** to `outputs/baseline_lock.json`; this
  notebook checks for that file before scoring and skips re-locking on a
  re-run, so the stated target for step 2 cannot silently move.
- **The minimum-positives-per-fold gate** (Section 4) ran automatically and
  is reported above (folds used vs. skipped), rather than assumed to always
  pass -- the early training years have naturally lower decline rates.
- **Every marginal AUC delta carries a quarter-clustered bootstrap 95% CI**
  (Section 7), resampling whole quarters rather than rows or counties, and
  the same resampled quarters are reused across the paired models being
  compared.
- **A CI including zero is reported as "no significant marginal
  contribution"** (Section 8), not rounded to a small positive result, even
  when the point estimate alone would have cleared the +0.01 threshold.
- **The interaction coefficient's sign was checked against
  `housing_risk.features.EXPECTED_INTERACTION_SIGN`** (Section 9) using a
  real fitted model, not just asserted.
- **The marginal lifts are order-dependent** (momentum first, then rate on
  top of it) -- they measure what rate contributes given momentum is
  already known, not rate's importance in isolation, per
  `docs/phase0_framing.md`.
- **Not done here:** the held-out test set is never touched in this
  notebook. Final scoring against it is notebook 10's job, after the
  robustness check (notebook 09) with a second algorithm family confirms
  (or doesn't) the same ranking of contributions.
